In [9]:

import pandas as pd
import numpy as np
import joblib
import tensorflow as tf
from pathlib import Path
from sklearn.preprocessing import RobustScaler, StandardScaler # Import both for flexibility
import traceback 
from tcn import TCN


In [13]:
# ADJUST THESE PARAMETERS TO MATCH THE TRAINING RUN THAT PRODUCED THE SAVED ARTIFACTS

# Data and Artifact Paths
base_dir = Path(".") 


NUM_FEATURES = 21      # Number of features AFTER selection + adding lagged features
NUM_OUTPUTS = 2        # Number of target variables (e.g., Offshore, Onshore)
SEQUENCE_LENGTH = 12   # Sequence length used during training
SCALER_TYPE = 'RobustScaler' 


VALIDATION_SPLIT_FRAC = 0.2 # Fraction used for train/validation split in training
start_date = '2020-01-01' # has to be the same as the training run
lag_hours = 1              # Lag hours used for lagged features in training
capacity_proxy_window = '365D' # Rolling window for capacity proxy
epsilon = 1e-6             # Epsilon added to capacity proxy

print(f"--- Configuration Set ---")
print(f"NUM_FEATURES: {NUM_FEATURES}")
print(f"NUM_OUTPUTS: {NUM_OUTPUTS}")
print(f"SEQUENCE_LENGTH: {SEQUENCE_LENGTH}")
print(f"SCALER_TYPE: {SCALER_TYPE}")
print(f"VALIDATION_SPLIT_FRAC: {VALIDATION_SPLIT_FRAC}")
print(f"START_DATE: {start_date}")
print(f"LAG_HOURS: {lag_hours}")
print(f"CAPACITY_PROXY_WINDOW: {capacity_proxy_window}")
print(f"Base directory: {base_dir.resolve()}")

--- Configuration Set ---
NUM_FEATURES: 21
NUM_OUTPUTS: 2
SEQUENCE_LENGTH: 12
SCALER_TYPE: RobustScaler
VALIDATION_SPLIT_FRAC: 0.2
START_DATE: 2020-01-01
LAG_HOURS: 1
CAPACITY_PROXY_WINDOW: 365D
Base directory: /Users/main/Developer/Tempestas/energy_price_prediction


In [17]:
# Define the exact list of features USED by the saved model
final_feature_cols = [
    'wind_speed_100m_offshore', 'surface_pressure_offshore', 'temperature_2m_offshore',
    'relative_humidity_2m_offshore', 'rain_offshore', 'hour_sin_offshore', 'hour_cos_offshore',
    'day_of_week_sin_offshore', 'day_of_week_cos_offshore', 'day_of_year_sin_offshore',
    'day_of_year_cos_offshore', 'wind_direction_100m_sin_offshore', 'wind_direction_100m_cos_offshore',
    'wind_speed_100m_onshore', 'temperature_2m_onshore', 'relative_humidity_2m_onshore',
    'rain_onshore', 'wind_direction_100m_sin_onshore', 'wind_direction_100m_cos_onshore',
    'Offshore_Norm_Lag1H', 'Onshore_Norm_Lag1H'
]

In [21]:
model_filename = f'best_tcn_model_{NUM_FEATURES}feat_reg.keras'
scaler_x_filename = f'scaler_x_{NUM_FEATURES}feat_{SCALER_TYPE}.joblib'
scaler_y_filename = f'scaler_y_{NUM_FEATURES}feat_{SCALER_TYPE}.joblib'
output_filename = f'stage1_predictions_mw_model_trained_from_{start_date}.csv'


model_path = base_dir / model_filename
scaler_x_path = base_dir / scaler_x_filename
scaler_y_path = base_dir / scaler_y_filename
output_csv_path = base_dir / output_filename

energy_data_path = base_dir / 'combined_total_energy_data_2017_2025.csv'
onshore_weather_path = base_dir / 'final_averaged_onshore_weather.csv'
offshore_weather_path = base_dir / 'final_averaged_offshore_weather.csv'

# Print paths to verify they are correct
print(f"\n--- Expected Artifacts ---")
print(f"Model: {model_path}")
print(f"Scaler X: {scaler_x_path}")
print(f"Scaler Y: {scaler_y_path}")
print(f"\n--- Expected Input Data ---")
print(f"Energy Data: {energy_data_path}")
print(f"Onshore Weather: {onshore_weather_path}")
print(f"Offshore Weather: {offshore_weather_path}")
print(f"\n--- Output File ---")
print(f"Predictions CSV: {output_csv_path}")



--- Expected Artifacts ---
Model: best_tcn_model_21feat_reg.keras
Scaler X: scaler_x_21feat_RobustScaler.joblib
Scaler Y: scaler_y_21feat_RobustScaler.joblib

--- Expected Input Data ---
Energy Data: combined_total_energy_data_2017_2025.csv
Onshore Weather: final_averaged_onshore_weather.csv
Offshore Weather: final_averaged_offshore_weather.csv

--- Output File ---
Predictions CSV: stage1_predictions_mw_model_trained_from_2020-01-01.csv


In [31]:
#Generates sequences from the data. Same implementation in the training script. Predicts the target at the end of the input sequence.


def create_sequences(X_data, y_data, sequence_length):
    
    X_seq_list, y_seq_list = [], []
    num_sequences_to_generate = len(X_data) - sequence_length
    print(f"Generating sequences. Input data length: {len(X_data)}, Sequence length: {sequence_length}")
    print(f"Number of sequences to generate: {num_sequences_to_generate}")

    for i in range(num_sequences_to_generate):
        sequence = X_data[i : i + sequence_length]
        target = y_data[i + sequence_length - 1]
        X_seq_list.append(sequence)
        y_seq_list.append(target)


    print(f"Generated {len(X_seq_list)} sequences.")
    return np.array(X_seq_list), np.array(y_seq_list)

print("create_sequences function defined.")


create_sequences function defined.


In [33]:
try:
    # Load the model
    loaded_model = tf.keras.models.load_model(model_path, custom_objects=custom_objects)
    print(f"Model loaded successfully from {model_path}")
    loaded_model.summary(print_fn=lambda x: print(f"Model Summary: {x}")) # Print summary

    # Load the scalers
    scaler_x = joblib.load(scaler_x_path)
    scaler_y = joblib.load(scaler_y_path)
    print(f"Scalers loaded successfully from {scaler_x_path} and {scaler_y_path}")
    print(f"Scaler X type: {type(scaler_x)}")
    print(f"Scaler Y type: {type(scaler_y)}")

except FileNotFoundError as e:
    print(f" loading files: {e}. Please ensure paths and filenames are correct and match the configuration.")
    print("Verify that the model and scaler files from the training run exist in the expected locations.")
    raise e # Re-raise the exception to stop notebook execution
except Exception as e:
    print(f"ERROR occurred during artifact loading: {e}")
    traceback.print_exc()
    raise e # Re-raise

print("\nArtifact loading completed.")

Model loaded successfully from best_tcn_model_21feat_reg.keras


Model Summary: Model: "TCN_21Feat_Reg_Compiled"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Input_Layer (InputLayer)        │ (None, 12, 21)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ TCN_21Feat_Reg (TCN)            │ (None, 16)             │        96,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Dense_Output_Regressor (Dense)  │ (None, 2)              │            34 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Linear_Output (Activation)      │ (None, 2)              │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘
 Total params: 286,840 (1.09 MB)
 Trainable params: 95,186 (371.82 KB)
 Non-trainable params: 1,280 (5.00

In [40]:
# Load & Preprocess Data (Load Raw, Merge, Basic Clean) 

energy_df = pd.read_csv(energy_data_path)
onshore_weather_df = pd.read_csv(onshore_weather_path)
offshore_weather_df = pd.read_csv(offshore_weather_path)
print(f"Raw data loaded shapes: Energy {energy_df.shape}, Onshore {onshore_weather_df.shape}, Offshore {offshore_weather_df.shape}")

energy_df['Timestamp'] = pd.to_datetime(energy_df['Timestamp (UTC)'], utc=True)
offshore_weather_df['Timestamp'] = pd.to_datetime(offshore_weather_df['date'], utc=True)
onshore_weather_df['Timestamp'] = pd.to_datetime(onshore_weather_df['date'], utc=True)

df_targets = energy_df.set_index('Timestamp').drop(columns=['Timestamp (UTC)'])
df_offshore_weather = offshore_weather_df.set_index('Timestamp').drop(columns=['date'])
df_onshore_weather = onshore_weather_df.set_index('Timestamp').drop(columns=['date'])

offshore_cols = {col: f"{col}_offshore" for col in df_offshore_weather.columns}
onshore_cols = {col: f"{col}_onshore" for col in df_onshore_weather.columns}
df_offshore_weather = df_offshore_weather.rename(columns=offshore_cols)
df_onshore_weather = df_onshore_weather.rename(columns=onshore_cols)


merged_df = pd.merge(df_targets, df_offshore_weather, left_index=True, right_index=True, how='inner')
final_df = pd.merge(merged_df, df_onshore_weather, left_index=True, right_index=True, how='inner')
print(f"Shape after merging: {final_df.shape}")

final_df = final_df.sort_index()
print(f"NaNs before initial fill: {final_df.isnull().sum().sum()}")
final_df = final_df.ffill().bfill() 
print(f"NaNs after initial fill: {final_df.isnull().sum().sum()}")


display(final_df.head()) 


Loading raw data files...
Raw data loaded shapes: Energy (72291, 3), Onshore (72312, 23), Offshore (72312, 23)
Applying datetime conversion, setting index, renaming columns...
Merging dataframes...
Shape after merging: (72290, 46)
NaNs before initial fill: 0
NaNs after initial fill: 0


,Wind_Offshore_MW,Wind_Onshore_MW,wind_speed_100m_offshore,wind_speed_10m_offshore,wind_gusts_10m_offshore,surface_pressure_offshore,temperature_2m_offshore,cloud_cover_low_offshore,cloud_cover_mid_offshore,cloud_cover_high_offshore,...,day_of_week_sin_onshore,day_of_week_cos_onshore,month_sin_onshore,month_cos_onshore,day_of_year_sin_onshore,day_of_year_cos_onshore,wind_direction_100m_sin_onshore,wind_direction_100m_cos_onshore,wind_direction_10m_sin_onshore,wind_direction_10m_cos_onshore
Timestamp,,,,,,,,,,,,,,,,,,,,,
2017-01-01 00:00:00+00:00,208.25,387.75,45.250201,34.425047,47.119998,1021.622222,7.093000,88.555556,0.111111,0.000000,...,-0.781831,0.62349,0.0,1.0,0.0,1.0,-0.626304,-0.770553,-0.537305,-0.831197
2017-01-01 01:00:00+00:00,219.00,381.75,45.455553,34.880133,48.919999,1021.188889,7.120778,76.666667,0.000000,0.000000,...,-0.781831,0.62349,0.0,1.0,0.0,1.0,-0.639300,-0.761805,-0.561538,-0.818629
2017-01-01 02:00:00+00:00,221.25,396.25,45.497017,34.937139,48.920000,1020.311111,6.998556,78.222222,1.111111,0.000000,...,-0.781831,0.62349,0.0,1.0,0.0,1.0,-0.641545,-0.755274,-0.562384,-0.812745
2017-01-01 03:00:00+00:00,223.00,413.25,45.676240,35.217506,49.239999,1019.600000,6.776333,85.777778,0.000000,1.222222,...,-0.781831,0.62349,0.0,1.0,0.0,1.0,-0.614654,-0.771571,-0.535464,-0.825089
2017-01-01 04:00:00+00:00,222.00,413.50,47.209270,36.723984,50.999999,1019.100000,6.504111,94.888889,0.222222,9.555556,...,-0.781831,0.62349,0.0,1.0,0.0,1.0,-0.600296,-0.785086,-0.529945,-0.832592


In [54]:
# Apply start_date filter 
rows_before_filter = final_df.shape[0]
final_df = final_df[final_df.index >= start_date]

print(f"Filtered dataframe shape: {final_df.shape} (removed {rows_before_filter - final_df.shape[0]} rows)")

# Calculate capacity_proxy 
print(f"Calculating capacity proxy (window: {capacity_proxy_window})...")
if 'Wind_Offshore_MW' in final_df.columns and 'Wind_Onshore_MW' in final_df.columns:
    final_df['Total_Wind_MW'] = final_df['Wind_Offshore_MW'] + final_df['Wind_Onshore_MW']
    proxy_base_col = 'Total_Wind_MW'
elif 'Wind_Offshore_MW' in final_df.columns:
    proxy_base_col = 'Wind_Offshore_MW'
elif 'Wind_Onshore_MW' in final_df.columns:
    proxy_base_col = 'Wind_Onshore_MW'
else:
    # Kept this critical check as calculation depends on it
    raise ValueError("Missing 'Wind_Offshore_MW' or 'Wind_Onshore_MW' columns for proxy calculation.")

final_df['capacity_proxy'] = final_df[proxy_base_col].rolling(window=capacity_proxy_window, min_periods=1).max()
final_df['capacity_proxy'] = final_df['capacity_proxy'].ffill().bfill() # Fill NaNs
final_df['capacity_proxy'] = final_df['capacity_proxy'] + epsilon # Add epsilon
print(f"Capacity proxy calculated using '{proxy_base_col}'.")
# print(final_df['capacity_proxy'].describe()) 

#  Calculate normalized targets 
target_cols_norm = []
if 'Wind_Offshore_MW' in final_df.columns:
    final_df['Offshore_Norm'] = (final_df['Wind_Offshore_MW'] / final_df['capacity_proxy']).clip(0, 1.1)
    target_cols_norm.append('Offshore_Norm')
if 'Wind_Onshore_MW' in final_df.columns:
    final_df['Onshore_Norm'] = (final_df['Wind_Onshore_MW'] / final_df['capacity_proxy']).clip(0, 1.1)
    target_cols_norm.append('Onshore_Norm')

print(f"Normalized targets created/updated: {target_cols_norm}")

# for col in target_cols_norm: print(final_df[col].describe())

# Add lagged normalized target features 
print(f"Adding lagged normalized targets (lag={lag_hours}H)...")
lag_col_offshore = f'Offshore_Norm_Lag{lag_hours}H'
lag_col_onshore = f'Onshore_Norm_Lag{lag_hours}H'
created_lag_cols = [] # Track which were actually created

if 'Offshore_Norm' in final_df.columns:
    final_df[lag_col_offshore] = final_df['Offshore_Norm'].shift(lag_hours)
    if lag_col_offshore in final_df.columns: created_lag_cols.append(lag_col_offshore)
if 'Onshore_Norm' in final_df.columns:
    final_df[lag_col_onshore] = final_df['Onshore_Norm'].shift(lag_hours)
    if lag_col_onshore in final_df.columns: created_lag_cols.append(lag_col_onshore)

if created_lag_cols:
    print(f"Lagged columns created/updated: {created_lag_cols}")

# Display potentially created features
display_cols = ['capacity_proxy'] + target_cols_norm + created_lag_cols
display(final_df[display_cols].head())

Filtered dataframe shape: (46010, 52) (removed 0 rows)
Calculating capacity proxy (window: 365D)...
Capacity proxy calculated using 'Total_Wind_MW'.
Normalized targets created/updated: ['Offshore_Norm', 'Onshore_Norm']
Adding lagged normalized targets (lag=1H)...
Lagged columns created/updated: ['Offshore_Norm_Lag1H', 'Onshore_Norm_Lag1H']


,capacity_proxy,Offshore_Norm,Onshore_Norm,Offshore_Norm_Lag1H,Onshore_Norm_Lag1H
Timestamp,,,,,
2020-01-01 00:00:00+00:00,235.500001,0.492569,0.507431,NaN,NaN
2020-01-01 01:00:00+00:00,235.500001,0.469214,0.523355,0.492569,0.507431
2020-01-01 02:00:00+00:00,242.500001,0.557732,0.442268,0.469214,0.523355
2020-01-01 03:00:00+00:00,249.250001,0.573721,0.426279,0.557732,0.442268
2020-01-01 04:00:00+00:00,249.250001,0.539619,0.435306,0.573721,0.426279


In [56]:
# Select Final Features and Target Columns 
X_all = final_df[final_feature_cols]
y_all = final_df[target_cols_norm]
print(f"Selected Features (X_all) shape: {X_all.shape}")
print(f"Selected Targets (y_all) shape: {y_all.shape}")

#  Split Data to Isolate Test Set 
print(f"Splitting data using VALIDATION_SPLIT_FRAC={VALIDATION_SPLIT_FRAC} to isolate test set...")
num_samples = len(X_all)

split_index = int(num_samples * (1 - VALIDATION_SPLIT_FRAC))
print(f"Total samples: {num_samples}, Split index: {split_index}")

X_test_df = X_all[split_index:].copy()
y_test_df = y_all[split_index:].copy()
capacity_proxy_test_period = final_df.loc[X_test_df.index, 'capacity_proxy'].copy()

X_test_df = X_test_df.ffill().bfill()


display(X_test_df.head())
display(y_test_df.head())
display(capacity_proxy_test_period.head())

Selected Features (X_all) shape: (46010, 21)
Selected Targets (y_all) shape: (46010, 2)
Splitting data using VALIDATION_SPLIT_FRAC=0.2 to isolate test set...
Total samples: 46010, Split index: 36808


,wind_speed_100m_offshore,surface_pressure_offshore,temperature_2m_offshore,relative_humidity_2m_offshore,rain_offshore,hour_sin_offshore,hour_cos_offshore,day_of_week_sin_offshore,day_of_week_cos_offshore,day_of_year_sin_offshore,...,wind_direction_100m_sin_offshore,wind_direction_100m_cos_offshore,wind_speed_100m_onshore,temperature_2m_onshore,relative_humidity_2m_onshore,rain_onshore,wind_direction_100m_sin_onshore,wind_direction_100m_cos_onshore,Offshore_Norm_Lag1H,Onshore_Norm_Lag1H
Timestamp,,,,,,,,,,,,,,,,,,,,,
2024-03-13 16:00:00+00:00,50.938066,1013.033333,9.587444,89.790078,0.011111,-0.866025,-5.000000e-01,0.974928,-0.222521,0.944489,...,-0.662766,-0.740341,34.643706,11.941786,79.516827,0.0,-0.795355,-0.597371,0.662762,0.269697
2024-03-13 17:00:00+00:00,52.030995,1012.688889,9.620778,89.036810,0.000000,-0.965926,-2.588190e-01,0.974928,-0.222521,0.944489,...,-0.660396,-0.745776,34.382293,11.591786,80.073983,0.0,-0.774284,-0.625850,0.660698,0.284234
2024-03-13 18:00:00+00:00,52.135877,1012.855556,9.698556,88.077025,0.000000,-1.000000,-1.836970e-16,0.974928,-0.222521,0.944489,...,-0.683815,-0.727978,34.136700,11.309643,80.562882,0.0,-0.732056,-0.674974,0.657917,0.278672
2024-03-13 19:00:00+00:00,50.544232,1012.766667,9.698556,88.199188,0.000000,-0.965926,2.588190e-01,0.974928,-0.222521,0.944489,...,-0.690381,-0.722289,33.978391,11.227500,80.195910,0.0,-0.699921,-0.712137,0.657116,0.283602
2024-03-13 20:00:00+00:00,49.638370,1012.622222,9.720778,88.199004,0.000000,-0.866025,5.000000e-01,0.974928,-0.222521,0.944489,...,-0.702438,-0.710216,33.677325,11.023929,80.549292,0.0,-0.693538,-0.718641,0.656274,0.276439


,Offshore_Norm,Onshore_Norm
Timestamp,,
2024-03-13 16:00:00+00:00,0.660698,0.284234
2024-03-13 17:00:00+00:00,0.657917,0.278672
2024-03-13 18:00:00+00:00,0.657116,0.283602
2024-03-13 19:00:00+00:00,0.656274,0.276439
2024-03-13 20:00:00+00:00,0.656274,0.278840


Timestamp
2024-03-13 16:00:00+00:00    5933.500001
2024-03-13 17:00:00+00:00    5933.500001
2024-03-13 18:00:00+00:00    5933.500001
2024-03-13 19:00:00+00:00    5933.500001
2024-03-13 20:00:00+00:00    5933.500001
Name: capacity_proxy, dtype: float64

In [58]:
# Scale Test Data using LOADED scalers 

X_test_scaled = scaler_x.transform(X_test_df)
y_test_scaled = scaler_y.transform(y_test_df)


# Create Sequences for Test Data 
X_test_seq, y_test_seq_generated = create_sequences(X_test_scaled, y_test_scaled, SEQUENCE_LENGTH)

#  Align Capacity Proxy with Test Sequences 
num_sequences_generated = X_test_seq.shape[0]
aligned_proxy_indices = range(SEQUENCE_LENGTH - 1, (SEQUENCE_LENGTH - 1) + num_sequences_generated)
capacity_proxy_test_seq_series = capacity_proxy_test_period.iloc[aligned_proxy_indices]
capacity_proxy_test_seq = capacity_proxy_test_seq_series.values.reshape(-1, 1)

# Get Timestamps for Predictions 
start_ts_index = SEQUENCE_LENGTH - 1
end_ts_index = start_ts_index + num_sequences_generated
prediction_timestamps = X_test_df.index[start_ts_index : end_ts_index]

print(f"Final shape for model input (X_test_seq): {X_test_seq.shape}")


Generating sequences. Input data length: 9202, Sequence length: 12
Number of sequences to generate: 9190
Generated 9190 sequences.
Final shape for model input (X_test_seq): (9190, 12, 21)


In [62]:
prediction_batch_size = 128 #match before
# `predict` returns numpy array
scaled_normalized_predictions = loaded_model.predict(X_test_seq, batch_size=prediction_batch_size, verbose=1)


# Verify the number of predictions matches the input sequences
if scaled_normalized_predictions.shape[0] != X_test_seq.shape[0]:
    raise ValueError(f"Shape mismatch: Number of predictions ({scaled_normalized_predictions.shape[0]}) doesn't match number of input sequences ({X_test_seq.shape[0]}).")
# Verify the number of output features matches configuration
if scaled_normalized_predictions.shape[1] != NUM_OUTPUTS:
    raise ValueError(f"Shape mismatch: Number of predicted outputs ({scaled_normalized_predictions.shape[1]}) doesn't match NUM_OUTPUTS ({NUM_OUTPUTS}). Model output structure might be wrong.")




72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step


In [78]:
# Inverse Transform & Convert to MW

# Inverse transform using scaler_y to get back to normalized scale (~0-1 range)
# scaler_y.inverse_transform expects shape (n_samples, n_features=NUM_OUTPUTS)
normalized_predictions = scaler_y.inverse_transform(scaled_normalized_predictions)

# Perform the element-wise multiplication. NumPy broadcasts (N, 1) proxy across the NUM_OUTPUTS columns.
actual_scale_predictions_mw = normalized_predictions * capacity_proxy_test_seq


temp_pred_df = pd.DataFrame(actual_scale_predictions_mw) # Temp df for easy stats
print(temp_pred_df.describe())



                 0            1
count  9190.000000  9190.000000
mean   1609.715831   786.741867
std    1261.698579   642.015847
min     -38.801672   -10.738222
25%     478.246645   263.515600
50%    1267.354702   563.451932
75%    2702.394895  1183.944151
max    4388.605943  2606.419874


In [83]:
#  Format and Save Output 

# Define column names based on NUM_OUTPUTS and expected order
# Ensure this order matches the target_cols_norm used for scaler_y fitting
if NUM_OUTPUTS == 2:
     output_columns = ['Predicted_Offshore_MW', 'Predicted_Onshore_MW']

# Clip negative predictions to zero (physical constraint)
neg_preds_before = (actual_scale_predictions_mw < 0).sum()
if neg_preds_before > 0:
     actual_scale_predictions_mw = np.maximum(actual_scale_predictions_mw, 0) # Clip inplace


# Create the final DataFrame
output_df = pd.DataFrame(
    actual_scale_predictions_mw,
    columns=output_columns,
    index=prediction_timestamps # Assign the determined timestamps
)
output_df.index.name = 'Timestamp (UTC)' # Name the index

print("\nFirst 5 rows of the prediction output (MW):")
display(output_df.head())
print("\nLast 5 rows of the prediction output (MW):")
display(output_df.tail())

# Save the DataFrame to CSV
print(f"\nSaving predictions to: {output_csv_path}...")
output_df.to_csv(output_csv_path, index=True) # index=True includes the timestamp index





First 5 rows of the prediction output (MW):


,Predicted_Offshore_MW,Predicted_Onshore_MW
Timestamp (UTC),,
2024-03-14 03:00:00+00:00,3205.506199,1575.157746
2024-03-14 04:00:00+00:00,3069.610038,1569.232811
2024-03-14 05:00:00+00:00,2818.686732,1482.163789
2024-03-14 06:00:00+00:00,2856.057539,1553.750631
2024-03-14 07:00:00+00:00,2907.658909,1554.390764



Last 5 rows of the prediction output (MW):


,Predicted_Offshore_MW,Predicted_Onshore_MW
Timestamp (UTC),,
2025-03-31 20:00:00+00:00,867.936831,248.843898
2025-03-31 21:00:00+00:00,828.444961,242.099983
2025-03-31 22:00:00+00:00,765.627246,227.248444
2025-03-31 23:00:00+00:00,764.214265,204.694980
2025-04-01 00:00:00+00:00,873.246187,201.278101



Saving predictions to: stage1_predictions_mw_model_trained_from_2020-01-01.csv...
